# Days 4-5: FastAPI Backend + Demo

**Goal**: Run the FastAPI backend exposed via ngrok and demo the search API.

**Note**: For Colab, Streamlit needs ngrok tunneling to be accessible from a browser.  
This notebook runs both the API and an inline Colab visual demo.

**Done checkpoint**:
- FastAPI running and accessible via public ngrok URL
- `POST /search` returns correct JSON results
- Visual demo works inline in this notebook

## 1. Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_DIR = '/content/drive/MyDrive/Projects/VisualSearchSystem'
LOCAL_DATA_DIR = '/content/data'

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f'CWD: {os.getcwd()}')

In [ ]:
import zipfile
from pathlib import Path

# Images must be on local disk — paths in images.csv and API demo both reference /content/data/raw/...
LOCAL_RAW_DIR = Path(LOCAL_DATA_DIR) / 'raw'
LOCAL_RAW_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR = LOCAL_RAW_DIR / 'images'

if IMAGES_DIR.exists() and any(IMAGES_DIR.iterdir()):
    print(f'Images on local disk: {len(list(IMAGES_DIR.glob("*.jpg"))):,} files.')
else:
    drive_zips = list((Path(PROJECT_DIR) / 'data' / 'raw').glob('*.zip'))
    if drive_zips:
        print('Extracting dataset from Drive zip to local disk...')
        with zipfile.ZipFile(drive_zips[0], 'r') as zf:
            zf.extractall(LOCAL_RAW_DIR)
        print(f'Done. {len(list(IMAGES_DIR.glob("*.jpg"))):,} images ready.')
    else:
        print('No zip found on Drive. Run Day 1 notebook first.')

In [ ]:
!pip install -q torch torchvision transformers faiss-cpu Pillow pandas numpy tqdm pyyaml \
    fastapi uvicorn[standard] python-multipart aiofiles pyngrok requests
print('Done.')

## 2. Load Config

In [ ]:
import yaml
from pathlib import Path

with open(f'{PROJECT_DIR}/configs/config.yaml') as f:
    config = yaml.safe_load(f)

BASE  = Path(PROJECT_DIR)
LOCAL = Path(LOCAL_DATA_DIR)

# Images read from local disk; all persistent files (index, embeddings) from Drive
config['paths']['raw_images']      = str(LOCAL / 'raw')
config['paths']['metadata_csv']    = str(BASE / 'data/metadata/images.csv')
config['paths']['embeddings_path'] = str(BASE / 'data/embeddings/embeddings.npy')
config['paths']['id_map_path']     = str(BASE / 'data/embeddings/id_map.json')
config['paths']['index_path']      = str(BASE / 'data/embeddings/faiss.index')
config['model']['device']          = 'auto'
config['api']['port']              = 8000

# Save updated config for the API subprocess
tmp_config = '/tmp/colab_config.yaml'
with open(tmp_config, 'w') as f:
    yaml.dump(config, f)
print('Config ready.')

## 3. Configure ngrok

In [ ]:
# Get a free ngrok token at: https://dashboard.ngrok.com/get-started/your-authtoken
# Then add it as a Colab Secret named NGROK_TOKEN

from google.colab import userdata
from pyngrok import ngrok, conf

try:
    ngrok_token = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(ngrok_token)
    print('ngrok token set from Colab Secrets.')
except Exception:
    print('NGROK_TOKEN not in Secrets. Set it manually:')
    print('  ngrok.set_auth_token("your_token_here")')
    # ngrok.set_auth_token('your_token_here')  # <-- uncomment and fill in

## 4. Start FastAPI Server in Background

In [ ]:
import subprocess
import time

# Write a startup script that patches config path
startup_script = f'''
import sys
sys.path.insert(0, "{PROJECT_DIR}")

import yaml
with open("{tmp_config}") as f:
    config = yaml.safe_load(f)

# Monkey-patch load_config in api.main to use our config
import src.api.main as api_main
api_main.load_config = lambda *args, **kwargs: config

import uvicorn
uvicorn.run(api_main.app, host="0.0.0.0", port=8000)
'''

with open('/tmp/start_api.py', 'w') as f:
    f.write(startup_script)

# Start server process
server = subprocess.Popen(['python', '/tmp/start_api.py'],
                          stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT)

print('Starting FastAPI server...')
time.sleep(15)  # Wait for model to load
print(f'Server PID: {server.pid}')

In [ ]:
# Open ngrok tunnel
from pyngrok import ngrok

# Kill any existing tunnels
ngrok.kill()

public_url = ngrok.connect(8000)
API_URL = str(public_url)
print(f'FastAPI is live at: {API_URL}')
print(f'Swagger docs:       {API_URL}/docs')

## 5. Test the API

In [ ]:
import requests

# Health check
r = requests.get(f'{API_URL}/health')
print(f'Health: {r.status_code}')
print(r.json())

In [ ]:
import pandas as pd

df = pd.read_csv(config['paths']['metadata_csv'])
sample = df.sample(1).iloc[0]
QUERY_PATH = sample['path']
print(f'Query: {sample.get("product_name", "")} | {sample.get("category", "")}')

# POST to /search
with open(QUERY_PATH, 'rb') as f:
    response = requests.post(
        f'{API_URL}/search',
        files={'file': ('query.jpg', f, 'image/jpeg')},
        params={'k': 10, 'mode': 'visual'},
        timeout=30
    )

print(f'Status: {response.status_code}')
data = response.json()
print(f'Results returned: {data["total"]}')
for r in data['results'][:3]:
    print(f'  ID={r["image_id"]} score={r["score"]:.4f} category={r.get("category")}')

## 6. Inline Visual Demo (No Browser Needed)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import requests, json

def colab_search_demo(query_path, k=12, mode='visual'):
    """Run search and display results as an inline grid."""
    with open(query_path, 'rb') as f:
        resp = requests.post(
            f'{API_URL}/search',
            files={'file': ('query.jpg', f, 'image/jpeg')},
            params={'k': k, 'mode': mode},
            timeout=30
        )
    results = resp.json()['results']

    cols = 4
    rows = (len(results) + 1 + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3.5))
    axes = axes.flatten()

    # Query
    axes[0].imshow(Image.open(query_path).resize((224,224)))
    axes[0].set_title('QUERY', color='royalblue', fontweight='bold', fontsize=9)
    axes[0].axis('off')

    # Results
    for i, r in enumerate(results):
        ax = axes[i+1]
        img_path = r.get('path')
        if img_path and Path(img_path).exists():
            ax.imshow(Image.open(img_path).resize((224,224)))
        else:
            ax.text(0.5, 0.5, f'ID:{r["image_id"]}', ha='center', va='center', transform=ax.transAxes)
        label = f'#{i+1} {r.get("category","")[:12]}\n{r.get("color","")[:10]}\n{r["score"]:.3f}'
        ax.set_title(label, fontsize=7)
        ax.axis('off')

    for ax in axes[len(results)+1:]:
        ax.axis('off')

    plt.suptitle(f'Visual Search — mode: {mode}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Run demo
colab_search_demo(QUERY_PATH, k=12)

In [ ]:
# Try 5 different random queries
for _ in range(5):
    sample = df.sample(1).iloc[0]
    colab_search_demo(sample['path'], k=8)

## 7. Streamlit App via ngrok (Optional — Opens in Browser)

In [ ]:
# Uncomment to run Streamlit with a public URL
# Requires ngrok configured above

# import subprocess, time
# from pyngrok import ngrok

# st_process = subprocess.Popen([
#     'streamlit', 'run', f'{PROJECT_DIR}/src/frontend/app.py',
#     '--server.port', '8501',
#     '--server.headless', 'true'
# ])

# time.sleep(5)
# st_url = ngrok.connect(8501)
# print(f'Streamlit is live at: {st_url}')

## ✅ Days 4-5 Done Checkpoint

In [ ]:
import requests

health = requests.get(f'{API_URL}/health').json()
assert health['status'] == 'ok', 'API health check failed'

with open(QUERY_PATH, 'rb') as f:
    resp = requests.post(
        f'{API_URL}/search',
        files={'file': ('query.jpg', f, 'image/jpeg')},
        params={'k': 10},
        timeout=30
    )
assert resp.status_code == 200, f'Search failed: {resp.text}'
assert resp.json()['total'] > 0

print('Days 4-5 COMPLETE')
print(f'  API URL:    {API_URL}')
print(f'  Docs:       {API_URL}/docs')
print(f'  Index size: {health["index_size"]:,}')